In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Load dataset
file_path = 'Modified_IT_Project_Team_Member_Recommendation_Data.csv'  # Replace with your dataset path
df = pd.read_csv(file_path)

# Separate features and target variable
X = df.drop('PerformanceScore', axis=1)
y = df['PerformanceScore']

# Preprocessing: Define which columns are numeric and which are categorical
numeric_features = ['Task_Duration', 'Member_Skill_Level', 'Member_Experience', 'Member_Workload']  # adjust as necessary
categorical_features = ['Task_Difficulty', 'Task_Type']  # adjust as necessary

# Create preprocessing pipelines
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Splitting data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Apply transformations
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)


In [3]:
import tensorflow as tf
from transformers import TFAutoModel, AutoTokenizer

# Example using BERT model, choose a model suitable for your task
model_name = 'bert-base-uncased'  # Replace with your chosen pre-trained model
tokenizer = AutoTokenizer.from_pretrained(model_name)
transformer_model = TFAutoModel.from_pretrained(model_name)

# Define input layers
input_ids = tf.keras.layers.Input(shape=(512,), name='input_ids', dtype='int32')
attention_mask = tf.keras.layers.Input(shape=(512,), name='attention_mask', dtype='int32')

# Define transformer model layers
embeddings = transformer_model(input_ids=input_ids, attention_mask=attention_mask)[0]

# Example of adding custom layers on top of transformer
x = tf.keras.layers.GlobalAveragePooling1D()(embeddings)
x = tf.keras.layers.Dense(10, activation='relu')(x)

# Output layer
output = tf.keras.layers.Dense(1, activation='linear')(x)  # Change activation if it's a classification task

# Compile the model
model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=[output])
model.compile(optimizer='adam', loss='mean_squared_error')  # Change loss if it's a classification task


/Users/binbasri0/anaconda3/envs/autoML/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PermissionError: [Errno 13] Permission denied: '/Users/binbasri0/.cache/huggingface/hub/models--bert-base-uncased'

In [ ]:

import tensorflow as tf
from transformers import TFAutoModel, AutoTokenizer
def encode_examples(X, tokenizer, max_length=512):
    # Tokenize the input (X)
    # X could be a single column or multiple columns concatenated into a single text
    return tokenizer.batch_encode_plus(
        X,
        max_len=max_length,
        padding='max_length',
        truncation=True,
        return_token_type_ids=False,
        return_attention_mask=True,
        return_tensors='tf'
    )

# Tokenize the training data
train_encodings = encode_examples(X_train, tokenizer)
test_encodings = encode_examples(X_test, tokenizer)

# Train the model
model.fit(
    {'input_ids': train_encodings['input_ids'], 'attention_mask': train_encodings['attention_mask']},
    y_train,
    validation_data=(
        {'input_ids': test_encodings['input_ids'], 'attention_mask': test_encodings['attention_mask']},
        y_test
    ),
    batch_size=32,
    epochs=3
)


FileNotFoundError: [Errno 2] No such file or directory: 'Modified_IT_Project_Team_Member_Recommendation_Data.csv'